In [ ]:
'''
%pip install pinecone-client pinecone-text
%pip install langchain-pinecone
'''

In [20]:
import numpy as
import pandas as pd
df = pd.read_csv('heritage_rag_full.csv')
df.head(1)

TypeError: Cannot convert numpy.ndarray to numpy.ndarray

In [2]:
import time
start = time.time()

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = CSVLoader('heritage_rag_full.csv', encoding='utf-8')
text_splitter = RecursiveCharacterTextSplitter( 
    chunk_size=1500, 
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

runtime = time.time() - start

print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 1.6269290447235107


In [3]:
len(document_list)

16280

In [6]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터 처음 업로드할때 사용

In [8]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()

index_name = "upstage-index"
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 19min 19s
Wall time: 59min 48s


# 업로드한 벡터DB 가져올때

In [9]:
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)

# 2. 답변 생성을 위한 Retrieval

In [14]:
query = "지금 삼성역에 있는데, 근처에 가볼만한 곳이 있어?"

질문입력ㄴ


In [10]:
# query = "지금 삼성역에 있는데, 근처에 가볼만한 곳이 있어?"
retriever = database.as_retriever(search_kwargs={'k':4})

# 3. 제공되는 prompt를 활용하여 답변 생성

In [11]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [12]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever, # database.as_retriever()
    chain_type_kwargs={"prompt":prompt}
)

In [17]:
ai_message = qa_chain.invoke({'query':input("질문입력")})
ai_message

질문입력광화문 사진 링크좀 줘


{'query': '광화문 사진 링크좀 줘',
 'result': '이곳은 광화문의 사진 링크입니다: [https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg](https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg)'}